# UTS DL – Task 3 (clustering multimodal) di Colab **atau** Kaggle
Notebook ini otomatis mendeteksi lingkungannya.

**Colab:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all**. Data diambil dari `data_uts_dl.zip` di Drive; output (checkpoint, log, grafik) disimpan di Drive (`OUT_DIR`). Jika sesi putus, **Run all** lagi: training dilanjutkan dari checkpoint epoch terakhir.

**Kaggle:** unggah `data_uts_dl.zip` sebagai *Dataset* pribadi lalu **Add Input** ke notebook ini. Settings: **Accelerator = GPU T4**, **Internet = On** (untuk `git clone` dan `pip`). Jalankan dengan **Save Version → Save & Run All (Commit)** agar tetap berjalan meski browser ditutup; hasil ada di tab *Output* (`/kaggle/working/uts_dl_outputs/hasil_task3.zip`). Untuk melanjutkan run yang terputus, tambahkan output versi sebelumnya sebagai input; sel konfigurasi menyalin checkpoint-nya ke `/kaggle/working`.

Perkiraan waktu di T4: SimCLR gambar 150 epoch + spektrum 100 epoch + embedding + diagnostik ≈ 30–50 menit (ETA tercetak per epoch). `MAX_MINUTES` menghentikan training dengan rapi sebelum batas sesi.

In [ ]:
import os, glob, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = not ON_KAGGLE and os.path.exists('/content')
print('Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'lokal')

# --- Sesuaikan bila perlu ---
REPO_URL = 'https://github.com/roplyhmn/UTS-Deep-Learning.git'
BRANCH   = 'task3/clustering'
MAX_MINUTES = 100        # berhenti rapi setelah N menit training (lanjut dengan menjalankan ulang)
RUN_EDA  = True          # grafik EDA 3a (~3 menit)

if ON_KAGGLE:
    WORK     = '/kaggle/working'
    OUT_DIR  = f'{WORK}/uts_dl_outputs'
    DATA_DIR = f'{WORK}/data'
    ZIP_PATH = None
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK     = '/content'
    OUT_DIR  = '/content/drive/MyDrive/uts_dl_outputs'
    DATA_DIR = '/content/data'
    ZIP_PATH = '/content/drive/MyDrive/data_uts_dl.zip'
REPO_DIR = f'{WORK}/UTS-Deep-Learning'
REPO_OUT = f'{OUT_DIR}/repo'                       # cermin struktur repo untuk hasil final
COMMON = (f'--data_dir {DATA_DIR} --out_dir {OUT_DIR} --pred_dir {REPO_OUT} '
          f'--results_dir {REPO_OUT}/results --models_dir {REPO_OUT}/models')
os.makedirs(REPO_OUT, exist_ok=True)

if ON_KAGGLE:   # lanjutkan run sebelumnya: output versi lama yang ditambahkan sebagai input
    for ck in glob.glob('/kaggle/input/**/uts_dl_outputs/task3/checkpoints', recursive=True):
        dst = f'{OUT_DIR}/task3/checkpoints'
        if not os.path.exists(dst):
            shutil.copytree(ck, dst); print('checkpoint lama disalin dari', ck)

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone -q -b $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch -q origin $BRANCH && git -C $REPO_DIR checkout -q $BRANCH && git -C $REPO_DIR reset -q --hard origin/$BRANCH
%cd $REPO_DIR
!git log --oneline -1
!pip install -q -r kode/requirements.txt

In [ ]:
# Data: Kaggle mengekstrak zip dataset otomatis (read-only) -> cari folder berisi clustering.npz;
# jika yang ada hanya zip, ekstrak. Colab: ekstrak zip dari Drive.
def flatten_zip(zp, dst):
    os.makedirs(dst, exist_ok=True)
    zipfile.ZipFile(zp).extractall(f'{WORK}/_unzip')
    for f in glob.glob(f'{WORK}/_unzip/**/*.*', recursive=True):
        shutil.move(f, os.path.join(dst, os.path.basename(f)))

if not os.path.exists(f'{DATA_DIR}/clustering.npz'):
    if ON_KAGGLE:
        hits = glob.glob('/kaggle/input/**/clustering.npz', recursive=True)
        if hits:
            os.makedirs(DATA_DIR, exist_ok=True)
            for f in glob.glob(os.path.join(os.path.dirname(hits[0]), '*')):
                if os.path.isfile(f):
                    os.symlink(f, os.path.join(DATA_DIR, os.path.basename(f)))
        else:
            zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
            assert zips, 'Dataset tidak ditemukan: Add Input -> dataset berisi data_uts_dl.zip'
            flatten_zip(zips[0], DATA_DIR)
    else:
        flatten_zip(ZIP_PATH, DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# Cek GPU + catat versi library (untuk laporan). Ditulis langsung dari kernel (bukan lewat !python
# dengan $OUT_DIR di dalam string shell, yang sebelumnya gagal menyimpan file ke Drive).
import sys, json, torch
assert torch.cuda.is_available(), 'GPU tidak aktif: Colab: Runtime -> T4 GPU; Kaggle: Accelerator -> GPU T4'
sys.path.insert(0, os.getcwd())
from kode.common.utils import env_info
ENV_INFO = env_info()
print(json.dumps(ENV_INFO, indent=1))
os.makedirs(OUT_DIR, exist_ok=True)
with open(f'{OUT_DIR}/env_info_task3.json', 'w') as f:
    json.dump(ENV_INFO, f, indent=1)

In [ ]:
# Uji Task 3b (CPU, ~10 detik): augmentasi, NT-Xent, resume identik, TTA
!python -m kode.task3.test_ssl

In [ ]:
# Uji cepat seluruh pipeline (3.000 sampel, 2 epoch; hasil ke OUT_DIR/task3/debug)
!python -m kode.task3.main --debug --no-eda --data_dir $DATA_DIR --out_dir $OUT_DIR

In [ ]:
# Task 3 PENUH: EDA (3a) -> SimCLR gambar + spektrum (3b), resume otomatis.
eda_flag = '--eda' if RUN_EDA else '--no-eda'
!python -m kode.task3.main {COMMON} {eda_flag} --max_minutes {MAX_MINUTES}

In [ ]:
# Ringkasan 3b
from IPython.display import Image, display
r = f'{REPO_OUT}/results/task3/ssl'
if os.path.exists(f'{r}/diagnostik_representasi.json'):
    print(json.dumps(json.load(open(f'{r}/diagnostik_representasi.json')), indent=1))
    for g in ['11_kurva_loss', '12_contoh_augmentasi', '13_diagnostik_representasi']:
        display(Image(f'{r}/{g}.png', width=900))
else:
    print('Belum selesai (batas MAX_MINUTES?) -> jalankan ulang notebook untuk melanjutkan.')

In [ ]:
# Paket hasil untuk repo: results/task3, models/ (+ embedding untuk tahap c-f)
os.makedirs(f'{REPO_OUT}/results/task3', exist_ok=True)
json.dump(ENV_INFO, open(f'{REPO_OUT}/results/task3/env_info.json', 'w'), indent=1)   # versi library run ini
shutil.make_archive(f'{OUT_DIR}/hasil_task3', 'zip', REPO_OUT)
shutil.make_archive(f'{OUT_DIR}/embedding_task3', 'zip', f'{OUT_DIR}/task3/ssl')
print('Unduh:', f'{OUT_DIR}/hasil_task3.zip', 'dan', f'{OUT_DIR}/embedding_task3.zip')

Setelah selesai: unduh `hasil_task3.zip` (grafik, json, bobot encoder) dan `embedding_task3.zip` (embedding `.npy`, log), lalu beri tahu Claude lokasinya.